[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/optics/lens_camera/lens_camera.ipynb)

# A Zoom Camera with Depth of Field

This example computes the sensor images and depth-of-field blur of a three-dimensional point scene photographed through a two-lens zoom camera. Points lying on the focused plane converge to sharp dots on the sensor, while points nearer or farther spread their light across elliptical blur discs that widen away from focus and at larger apertures. Translating the rear lens changes magnification, shifting the sensor refocuses, and tilting the sensor brings an oblique Scheimpflug plane into sharp focus across depth layers.

An ideal thin lens is modelled as a projective collineation on points, `Point + origin * (home & Point) / focal`, whose induced map on lines `Line - (origin & (Line ^ home)) / focal` equals the join of the point images so a multi-lens train composes identically on points and on rays. The rays from a scene point through the circular pupil form an aperture cone obtained by pulling the pupil ball back through central projection `(scene & Point) ^ front_plane` via `.adjugate()`; pulling that cone forward through `collineation.inverse()` gives the image cone directly, and intersecting with the sensor plane cuts out the exact blur conic without solving for intermediate focal planes.

In homogeneous matrix projective geometry, each thin lens is assembled as a $4 \times 4$ coordinate homography $H_k = R_k (I + f_k^{-1} e_4 e_1^{\mathsf T}) R_k^{-1}$ while ray transfers require separate $6 \times 6$ Plücker line matrices $\Lambda(H_k)$, central projection from scene point $X$ onto pupil plane $\pi$ builds the rank-3 matrix $P_X = (\pi^{\mathsf T} X) I - X \pi^{\mathsf T}$ to pull the $4 \times 4$ pupil sphere $Q$ back as $C = P_X^{\mathsf T} Q P_X$, and slicing the pushed cone $H^{-\mathsf T} C H^{-1}$ on a tilted sensor frame $M$ extracts a $3 \times 3$ conic matrix $E^{\mathsf T} M^{\mathsf T} H^{-\mathsf T} C H^{-1} M E$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from examples.optics.lens_camera import render, scenarios
from examples.util.animation import save_figure, show_states
from numga import Algebra, NumpyContext, dim, stack
from numga.algebras import PGA3D as ga
from numga.gatype.traits import Versor

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga, units=True).multivector
Scalar = ga.gatype.scalar()                                                     # Scalar+
Point = ga.gatype.antivector()                                                  # Antivector-
Plane = ga.gatype.vector()                                                      # Vector+
SensorPlane = Plane.derive.within(Algebra("w0y+z+"))                            # Vector+
SensorPoint = SensorPlane.derive.dual                                           # Antivector-
# Lines are joins of points.
Line = ga.gatype.antibivector()                                                 # Antibivector-
Motor = ga.gatype.rotor()                                                       # Rotor+
PointMap = ga.gatype((Point, Point))                                            # Point- <- Point-
LineMap = ga.gatype((Line, Line))                                               # Line- <- Line-
Quadric = ga.gatype((Plane, Point))                                             # Plane+ <- Point-
Camera = ga.gatype((Point, Point, Point))                                       # Point- <- (Point-, Point-)


def unit_point(points: Point) -> Point:
    """Dividing by point weight and normalizing gives the positive unit-weight versor representative (`[...] Point-`)."""
    return (points / (mv.w & points)).normalized().with_traits(Versor)


# Every optical element is defined in its home frame, centred on the origin in the plane x == 0.
origin = unit_point(mv.w.dual())                                                # [] Point-
home = mv.x                                                                     # [] Plane+

## Thin-lens collineations and pulled-back aperture cones

In its home plane, an ideal thin lens fixes every point on `home` and shifts any off-plane point along its line toward `origin` in proportion to `home & Point` over the focal length, while on lines it subtracts `origin & (Line ^ home)` over the focal length. Conjugating by each element's placement motor positions the front and rear lenses, and pulling the front pupil ball back through central projection from each scene point and forward through the inverse train collineation produces every scene point's image cone at once.

In [ ]:
def thin_lens_maps(focal: Scalar) -> tuple[PointMap, LineMap]:
    """Focal lengths `focal` (`[n_lenses] Scalar+, length`) in the home plane define point collineations (`[n_lenses] Point- <- Point-`) and induced line maps (`[n_lenses] Line- <- Line-`)."""
    point_maps = Point + origin * (home & Point) / focal                        # [...] Point- <- Point-
    line_maps = Line - (origin & (Line ^ home)) / focal                         # [...] Line- <- Line-
    return point_maps, line_maps


def aperture_ball(radius: Scalar) -> Quadric:
    """Radius `radius` (`[] Scalar+, length`) around `origin` defines a spherical primal quadric (`[] Plane+ <- Point-`) whose home-plane section bounds the aperture."""
    coordinate_planes = stack((mv.x, mv.y, mv.z))                               # [3] Plane+
    return (coordinate_planes * (coordinate_planes & Point)).sum(-1) - mv.w * (mv.w & Point) * radius**2

In [ ]:
def camera_exposure(
    scene: Point,
    subject: Point,
    focal: Scalar,
    placements: Motor,
    focus: Point,
    tilt: Motor,
    radius: Scalar,
    rim: Point,
) -> tuple[PointMap, Camera, Motor, Quadric, Plane, Point]:
    """Exposing `scene` (`[...] Point-`) through the two lenses at `placements` (`[2] Rotor+`) with aperture `radius`, focus target `focus`, and sensor `tilt` produces the train collineation, camera map, sensor frame, image cones (`[...] Plane+ <- Point-`), element planes (`[3] Plane+`), and traced ray vertices (`[4, ...] Point-`)."""
    # Place the front and rear lenses, then compose their point collineations and line maps.
    lens_points, lens_lines = thin_lens_maps(focal)                             # [2] Point- <- Point-, Line- <- Line-
    point_maps = placements >> lens_points(placements << Point)                 # [2] Point- <- Point-
    line_maps = placements >> lens_lines(placements << Line)                    # [2] Line- <- Line-
    front, rear = line_maps[0], line_maps[1]                                    # [] Line- <- Line-, Line- <- Line-
    front_plane, rear_plane = placements[0] >> home, placements[1] >> home      # [] Plane+, Plane+
    collineation = point_maps[1](point_maps[0])                                 # [] Point- <- Point-
    train = rear(front)                                                         # [] Line- <- Line-
    pupil_ball = placements[0] >> aperture_ball(radius)(placements[0] << Point)  # [] Plane+ <- Point-

    # The image of the focus target places the sensor centre, and `tilt` turns the sensor plane about that image.
    image = unit_point(collineation(focus))                                     # [] Point-
    frame = (image / origin).square_root() * tilt                               # [] Rotor+
    sensor = frame >> home                                                      # [] Plane+
    camera = train(Point & Point) ^ sensor                                      # [] Point- <- (Point-, Point-)

    # Pull the pupil ball back through central projection from each scene point, then push forward through the lens train.
    projection = (scene & Point) ^ front_plane                                  # [...] Point- <- Point-
    aperture_cones = projection.adjugate()(pupil_ball(projection))              # [...] Plane+ <- Point-
    inverse_collineation = collineation.inverse()                               # [] Point- <- Point-
    image_cones = inverse_collineation.adjugate()(aperture_cones(inverse_collineation))  # [...] Plane+ <- Point-

    rays = subject & (placements[0] >> rim)                                     # [...] Line-
    ray_vertices = stack([
        subject.broadcast_to(rays.shape),
        unit_point(rays ^ front_plane),
        unit_point(front(rays) ^ rear_plane),
        unit_point(rear(front(rays)) ^ sensor),
    ])                                                                          # [4, ...] Point-, 1 | length
    return collineation, camera, frame, image_cones, stack([front_plane, rear_plane, sensor]), ray_vertices

## Wide, telephoto, and Scheimpflug sensor tilt

Focusing on the middle depth layer renders that layer as sharp dots while spreading the nearer and farther layers into blur discs; moving the rear lens back into the telephoto position magnifies the field and widens out-of-focus blur. Tilting the sensor about the focus image inclines the plane of sharp focus in object space so opposite vertical edges of the near and far layers come into focus simultaneously.

In [ ]:
# Front and rear focal lengths; the front lens stands fixed at x == 1 while the rear lens moves to zoom.
focal_lengths = mv.scalar([[1.0], [0.6]], unit=dim.length)
front_at = mv.scalar(1.0, unit=dim.length)
rim_heights = np.array([1, 0, -1])
# Wide, telephoto, and tilted-sensor still settings.
wide_at, tele_at = mv.scalar(1.4, unit=dim.length), mv.scalar(1.8, unit=dim.length)
focus_distance = mv.scalar(2.2, unit=dim.length)
aperture = mv.scalar(0.45, unit=dim.length)
sensor_tilt = np.radians(25)

In [ ]:
def camera_setting(
    rear_at: Scalar, focus_at: Scalar, tilt: float, radius: Scalar,
) -> tuple[PointMap, Camera, Motor, Quadric, Plane, Point]:
    """Exposing `scenarios.scene` with the rear lens at `rear_at`, focus target at `focus_at`, sensor tilt `tilt`, and aperture `radius` evaluates `camera_exposure`."""
    placements = (mv.xw * (stack((front_at, rear_at)) / 2)).exp()               # [2] Rotor+, 1 | length
    focus = origin + mv.x.dual() * focus_at                                     # [] Point-, 1 | length
    turn = (mv.xy * (tilt / 2)).exp()                                           # [] Rotor+
    rim = origin + mv.y.dual() * (radius * rim_heights)                         # [3] Point-, 1 | length
    return camera_exposure(scenarios.scene, scenarios.subject, focal_lengths, placements, focus, turn, radius, rim)


wide = camera_setting(wide_at, -focus_distance, 0.0, aperture)
tele = camera_setting(tele_at, -focus_distance, 0.0, aperture)
tilted = camera_setting(tele_at, -focus_distance, sensor_tilt, aperture)
save_figure(render.stills_figure((wide, tele, tilted)), "lens_camera");

From left to right, the wide-angle exposure keeps the green middle depth layer sharp between soft red (far) and blue (near) blur discs, the telephoto exposure magnifies the grid and enlarges out-of-focus discs, and the tilted-sensor exposure sweeps a slanted plane of sharp focus across red at the bottom, green across the middle, and blue at the top.

## Blur conics from image-cone sections and pupil homographies

Slicing a scene point's image cone along the sensor plane from the chief-ray intersection traces the boundary of its blur conic. Pushing the front pupil's flat dual quadric directly through the point's pupil-to-sensor map `camera(far_corner)` produces the dual conic of the identical ellipse, and at an in-focus point the traced section collapses onto the chief-ray hit.

In [ ]:
def cone_section(cone: Quadric, start: Point, frame: Motor, n_samples: int) -> Point:
    """Tracing rays outward from interior point `start` along `n_samples` sensor-plane directions samples the conic boundary (`[n_samples] Point-`) where `cone` meets the sensor."""
    angles = np.linspace(0.0, 2 * np.pi, n_samples, endpoint=False)
    turns = (mv.yz * (-angles / 2)).exp()                                       # [n_samples] Rotor+
    sensor_directions = frame >> (turns >> mv.y.dual())                         # [n_samples] Point-
    # Evaluating the quadric form along `start + sensor_directions * step` yields a scalar quadratic in `step`.
    a = cone(sensor_directions) & sensor_directions                             # [n_samples] Scalar+
    b = cone(start) & sensor_directions                                         # [n_samples] Scalar+
    c = cone(start) & start                                                     # [] Scalar+
    # Clamping suppresses tiny negative rounding residuals at a sharp focus vertex where the section collapses to `start`.
    root = (b * b - a * c).clip(0.0, np.inf).square_root()                      # [n_samples] Scalar+
    return start + sensor_directions * ((root - b) / a)                         # [n_samples] Point-


n_samples = 48

collineation, camera, frame, cones, _, _ = tilted
place_front = (mv.xw * (front_at / 2)).exp()                                    # [] Rotor+
sensor, centre = frame >> home, place_front >> origin                           # [] Plane+, Point-
pupil_directions = stack((mv.y.dual(), mv.z.dual()))                            # [2] Point-
aperture_disc = (pupil_directions * (Plane & pupil_directions)).sum(-1) * aperture**2 - origin * (Plane & origin)  # [] Point- <- Plane+
pupil = place_front >> aperture_disc(place_front << Plane)                      # [] Point- <- Plane+
far_corner = scenarios.scene[2, 4, 3]                                           # [] Point-
start = (collineation(far_corner) & collineation(centre)) ^ sensor              # [] Point-
boundary = cone_section(cones[2, 4, 3], start, frame, n_samples)                # [n_samples] Point-
pupil_to_sensor = camera(far_corner)                                            # [] Point- <- Point-
blur_disc = pupil_to_sensor(pupil(pupil_to_sensor.adjugate()))                  # [] Point- <- Plane+
home_blur_disc = (frame << blur_disc(frame >> SensorPlane)).cast(SensorPoint)   # [] Point- <- Plane+
hits = (frame << boundary).cast(SensorPoint)                                    # [n_samples] Point-

## Continuous zoom, focus sweep, and aperture stop-down

Oscillating the rear lens position zooms the field of view while sweeping the focus distance across the three depth layers and opening and closing the front aperture.

In [ ]:
rear_middle, rear_swing = mv.scalar(1.6, unit=dim.length), mv.scalar(0.2, unit=dim.length)
focus_middle, focus_swing = mv.scalar(2.4, unit=dim.length), mv.scalar(0.8, unit=dim.length)
aperture_middle, aperture_swing = mv.scalar(0.3, unit=dim.length), mv.scalar(0.1, unit=dim.length)
n_frames = 108
frame_duration_ms = 40

In [ ]:
def zoom_states(n_frames: int) -> Iterator[tuple]:
    """Yields `(rear_at, focus_at, radius, exposure)` across `n_frames` phases as the camera zooms, refocuses, and stops down."""
    for phase in np.linspace(0.0, 2 * np.pi, n_frames, endpoint=False):
        rear_at = rear_middle + rear_swing * np.sin(phase)
        focus_at = focus_middle - focus_swing * np.cos(phase)
        radius = aperture_middle + aperture_swing * np.sin(2 * phase)
        yield rear_at, focus_at, radius, camera_setting(rear_at, -focus_at, 0.0, radius)


recorded = show_states(
    zoom_states(n_frames),
    render.camera_animation,
    scenarios.scene,
    name="lens_camera",
    duration_ms=frame_duration_ms,
    n_frames=n_frames,
);

The upper panel renders the live sensor image while the lower three-dimensional view shows the red, green, and blue object layers, the front aperture lens and sliding rear zoom lens in translucent blue, the dark sensor plate shifting to track the focus plane, and three orange rays from one red subject point refracting through both lenses onto the sensor.

In [ ]:
# checks
# Every image cone has its vertex at the collineated scene point, the traced sensor section lies on the
# pushed-forward aperture dual conic, and an in-focus point collapses its blur section onto the chief-ray hit.
assert ((cones(collineation(scenarios.scene)) & collineation(scenarios.scene)).abs() < mv.scalar(1e-9, unit=dim.length**4)).all()
assert ((home_blur_disc.solve(hits) & hits).abs() < 1e-9).all()

wide_collineation, _, wide_frame, wide_cones, _, _ = wide
focused_hit = (wide_collineation(scenarios.scene[1, 2, 1]) & wide_collineation(centre)) ^ (wide_frame >> home)  # [] Point-
focused_section = unit_point(cone_section(wide_cones[1, 2, 1], focused_hit, wide_frame, n_samples))  # [n_samples] Point-
assert ((focused_section & unit_point(focused_hit)).norm() < mv.scalar(1e-6, unit=dim.length)).all()
assert len(recorded) == n_frames